In [88]:
import heapq #El modulo heapq implementa colas de prioridad (heaps)

In [89]:
class Node:
    def __init__(self, position, parent=None,action=None, path_cost=0): #AGREGAR ACTION
        self.position = position
        self.parent = parent
        self.action = action          # Movimiento (ej. 'UP', 'DOWN', etc.)
        self.path_cost = path_cost

    def __lt__(self, other):
        return self.path_cost < other.path_cost

class Problem:

    def __init__(self, initial, goal, actions, result, action_cost, is_goal):
        self.initial = initial
        self.goal = goal
        self.actions = actions
        self.result = result
        self.action_cost = action_cost
        self.is_goal = is_goal

In [90]:
def reconstruct_path(node):
    path = []
    cost = node.path_cost  # ← Aquí guardas el costo real

    while node:
        path.append((node.action, node.position))
        node = node.parent
    path.reverse()

    print("Total cost:", cost)
    return path

In [91]:
def find_exit(maze):
    start = (1, 1)  # Posicion inicial
    end = (1, 6)    # Posicion de la salida

    # Conjunto de acciones posibles
    MOVES = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}

    # Funciones necesarias para definir Problem
    def actions(state):
        row, col = state
        possible = []
        for move, (dr, dc) in MOVES.items():
            nr, nc = row + dr, col + dc
            if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]):
                if maze[nr][nc] != "#":
                    possible.append(move)
        return possible

    def result(state, action):
        dr, dc = MOVES[action]
        return (state[0] + dr, state[1] + dc)

    # def action_cost(state, action):
    #     if action in ("LEFT", "RIGHT"): #en este caso moverse a los laterales cuesta mas
    #         return 2
    #     return 1
    
    #def action_cost(state,action): # caso donde todo movimiento vale 1
       # return 1
    def action_cost(state, action):
        row, col = result(state, action)# caso donde existe terreno irregular y cuesta mas moverse
        if maze[row][col] == "x":
            return 2  # Terreno irregular
        return 1      # Camino normal

    def is_goal(state):
        return state == end

    # Crear el objeto Problem
    problem = Problem(start, end, actions, result, action_cost, is_goal)

    # Heurística de Manhattan
    def manhattan_distance(pos, goal):
        return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])

    # Algoritmo A*
    start_node = Node(start, path_cost=0, action=None)
    frontier = [(manhattan_distance(start, end), start_node)]
    heapq.heapify(frontier)
    reached = {start: start_node}

    while frontier:
        _, node = heapq.heappop(frontier)
        if node.position == end:
            return reconstruct_path(node)  # reconstruccion con acciones

        for action in problem.actions(node.position):
            neighbor = problem.result(node.position, action)
            new_cost = node.path_cost + problem.action_cost(node.position, action)
            if neighbor not in reached or new_cost < reached[neighbor].path_cost:
                reached[neighbor] = Node(neighbor, parent=node, action=action, path_cost=new_cost)
                heapq.heappush(
                    frontier,
                    (new_cost + manhattan_distance(neighbor, end), reached[neighbor]),
                )

    return None  # No se encontro salida

In [92]:
maze = [
    ["#", "#", "#", "#", "#", "#", "#", "#", "#", "#"],
    ["#", "S", " ", " ", "#", " ", "x", " ", "E", "#"],
    ["#", "#", "#", " ", "#", " ", "#", "x", "#", "#"],
    ["#", " ", " ", " ", " ", " ", "#", " ", " ", "#"],
    ["#", "x", "#", "#", "#", " ", "#", "#", "x", "#"],
    ["#", " ", " ", "#", " ", " ", " ", " ", " ", "#"],
    ["#", "#", "#", "#", "#", "#", "#", "#", "#", "#"],
]

path = find_exit(maze)
print("Path to exit:", path)

Total cost: 10
Path to exit: [(None, (1, 1)), ('RIGHT', (1, 2)), ('RIGHT', (1, 3)), ('DOWN', (2, 3)), ('DOWN', (3, 3)), ('RIGHT', (3, 4)), ('RIGHT', (3, 5)), ('UP', (2, 5)), ('UP', (1, 5)), ('RIGHT', (1, 6))]
